In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="ymoslem/Wikimedia-Speech-Irish", 
    repo_type="dataset", local_dir="./Wikimedia-Speech-Irish", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 9 files: 100%|██████████| 9/9 [00:03<00:00,  2.83it/s]


'/home/ubuntu/Wikimedia-Speech-Irish'

In [3]:
files = glob('Wikimedia-Speech-Irish/*/*.parquet')
len(files)

9

In [4]:
df = pd.read_parquet(files[0])
df

,audio,text_ga,text_en
0,{'bytes': b'RIFFv(\x05\x00WAVEfmt \x12\x00\x00...,Bíonn an codarsnacht mór seo le feiceáil ar ai...,The contrasts greatly reflected on the archite...
1,{'bytes': b'RIFF\x16\x0c\x04\x00WAVEfmt \x12\x...,Féadann táscairí athrú in airíonna fisiciúla e...,Indicators can also show change in other physi...
2,{'bytes': b'RIFF\x16\xf9\x02\x00WAVEfmt \x12\x...,D'áitigh máthair dhosháraithe Muhammad XII an ...,The indomitable mother of Muhammad XII insiste...
3,{'bytes': b'RIFF\xa6K\x02\x00WAVEfmt \x12\x00\...,D’fhéadfaí a mheas gur tionscnóir na ceimice r...,He may be regarded as the initiator of modern ...
4,{'bytes': b'RIFF&\xee\x02\x00WAVEfmt \x12\x00\...,Fuair sé amach go raibh “céasadh meabhrach agu...,"It found that there was ""mental and physical t..."
...,...,...,...
1671,{'bytes': b'RIFFV\xf3\x04\x00WAVEfmt \x12\x00\...,"Bhí Baile Dúill, lena suíomh cois uisce dhídea...",Baldoyle with its sheltered waterside location...
1672,{'bytes': b'RIFF\x06#\x02\x00WAVEfmt \x12\x00\...,Is í céile Phríomh-Aire na Cóiré Theas Dara Be...,The wife of the Prime Minister of South Korea ...
1673,{'bytes': b'RIFFfE\x02\x00WAVEfmt \x12\x00\x00...,Ba í an t-iriseoir Eibhlín Ní Bhriain an t-aon...,The couple's only child was the journalist Eib...
1674,{'bytes': b'RIFF\x16\xc7\x02\x00WAVEfmt \x12\x...,Bráithreachas Protastúnach Uladh agus aontacht...,The Orange Order is an Ulster Protestant and u...


In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text_ga'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 1/1 [01:25<00:00, 85.60s/it]


In [7]:
len(data)

15090

In [8]:
data[0]

{'audio_filename': 'Wikimedia-Speech-Irish_audio/Wikimedia-Speech-Irish-data-train-00006-of-00009_0.mp3',
 'text': 'Bíonn an codarsnacht mór seo le feiceáil ar ailtireacht na cathrach, áit ina mbíonn árais sócúlacha agus monarchana deargbhriceach agus seantithe tionóntáin taobh le chéile.',
 'speaker': 'Wikimedia-Speech-Irish_audio'}

In [9]:
with open('Wikimedia-Speech-Irish.json', 'w') as fopen:
    json.dump(data, fopen)

In [10]:
audio_files = [d['audio_filename'] for d in data]

with open('Wikimedia-Speech-Irish-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [13]:
# !zip -rq Wikimedia-Speech-Irish_audio.zip Wikimedia-Speech-Irish_audio

In [14]:
# !hf upload malaysia-ai/Multilingual-TTS Wikimedia-Speech-Irish_audio.zip --repo-type=dataset

In [15]:
import json

with open('Wikimedia-Speech-Irish.json') as fopen:
    rows = json.load(fopen)

mapping = {}
for i in tqdm(range(len(rows))):
    mapping[rows[i]['audio_filename']] = i
len(mapping)

100%|██████████| 15090/15090 [00:00<00:00, 3561335.10it/s]


15090

In [16]:
import faiss
import os
import numpy as np
from tqdm import tqdm

data = {}
d = 192
index = faiss.IndexFlatL2(d)

centroids = []

def assign(x, threshold=0.1):
    if len(centroids) == 0:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return 0
    
    D, I = index.search(np.array([x], dtype=np.float32), 1)
    if D[0][0] > threshold:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return len(centroids)-1
    else:
        return I[0][0]
        
for i in tqdm(range(len(rows))):
    index_ = mapping[rows[i]['audio_filename']]
    v_f = f'Wikimedia-Speech-Irish_embedding/{index_}.npy'
    if not os.path.exists(v_f):
        continue
    try:
        v = np.load(v_f)
        data[rows[i]['audio_filename']] = assign(v)
    except Exception as e:
        pass

100%|██████████| 15090/15090 [00:02<00:00, 5425.35it/s]


In [17]:
for i in range(len(rows)):
    s = data[rows[i]['audio_filename']]
    rows[i]['speaker'] = rows[i]['speaker'] + f'_{s}'

In [18]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

{'audio_filename': 'Wikimedia-Speech-Irish_audio/Wikimedia-Speech-Irish-data-train-00006-of-00009_0.mp3',
 'text': 'Bíonn an codarsnacht mór seo le feiceáil ar ailtireacht na cathrach, áit ina mbíonn árais sócúlacha agus monarchana deargbhriceach agus seantithe tionóntáin taobh le chéile.',
 'speaker': 'Wikimedia-Speech-Irish_audio_0'}

In [19]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'Wikimedia-Speech-Irish')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 124.82ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  778kB /  778kB,  130kB/s  
Processing Files (1 / 1): 100%|██████████|  778kB /  778kB,  125kB/s  
New Data Upload: 100%|██████████|  778kB /  778kB,  125kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:06<00:00,  6.52s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/cd8805ac73b6ee8f2cb667c02488cb751dd43440', commit_message='Upload dataset', commit_description='', oid='cd8805ac73b6ee8f2cb667c02488cb751dd43440', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [22]:
# !zip -rq Wikimedia-Speech-Irish_audio_neucodec.zip Wikimedia-Speech-Irish_audio_neucodec

In [23]:
# !hf upload malaysia-ai/Multilingual-TTS Wikimedia-Speech-Irish_audio_neucodec.zip --repo-type=dataset